# WMX 로그 기반 Newton Motor Parameter Fitting

WMX 실측 로그의 `Command Position`, `Feedback Position`, `Feedback Velocity`, `Feedback Torque`를 기준으로 1-DOF Newton 서보 모터 모델의 파라미터를 자동 추정한다.

이 Notebook의 처리 순서는 다음과 같다.

1. fitting용 WMX 로그를 읽고 단위와 샘플 주기를 확인한다.
2. `differential_evolution`으로 넓은 범위에서 전역 탐색한다.
3. 전역 탐색 결과를 `L-BFGS-B`의 시작점으로 사용해 연속 파라미터를 정밀 조정한다.
4. fitting에 사용하지 않은 별도 WMX 로그로 일반화 성능을 검증한다.

`01_newton_single_motor_model.ipynb`의 `MotorParams`와 `simulate_motor()`를 재사용한다. 02의 시험 지령 생성 함수는 실측 Command를 그대로 사용하므로 필요하지 않다. 03은 로그 경로 설정과 비교 코드까지 실행되는 Notebook이므로 전체를 `%run`하지 않고, 이 파일에 fitting용 로그 처리와 평가 함수를 부작용 없이 정의한다.

## 1. 식별 전에 알아둘 점

파라미터 fitting은 단순히 최적화 함수를 호출하는 작업이 아니라, **로그가 각 파라미터를 충분히 드러내는가**가 중요한 시스템 식별 문제이다.

- 작은 진폭의 저속 운동만 있으면 `effort_limit`, `coulomb`, `inertia`를 구분하기 어렵다.
- `ki`와 `integral_max`는 오차가 일정 시간 누적되는 구간이 있어야 관측하기 쉽다.
- 가속·감속 구간은 `inertia`, 일정 속도 구간은 `viscous`와 `coulomb`, 추종 과도응답은 `kp`, `kd`, `delay_steps` 식별에 특히 중요하다.
- WMX Torque가 실제 `N·m`인지, 정격 토크 대비 `%`인지 먼저 확인해야 한다. `%`라면 아래 `torque_scale`로 `N·m` 변환 계수를 적용한다.
- WMX timestamp가 제어 계산 전 상태인지 후 상태인지에 따라 한 sample 오프셋이 생길 수 있다. 03의 파형 비교로 먼저 시간 정렬을 확인한다.

한 로그로 loss가 작아져도 진짜 물리 파라미터를 찾았다고 단정할 수 없다. 서로 다른 command 특성을 가진 별도 로그에서 성능이 유지되는지 반드시 확인한다.

In [ ]:
%run 01_newton_single_motor_model.ipynb

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution, minimize

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "code" / "wmx_log_utils.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("code/wmx_log_utils.py를 찾을 수 없습니다.")
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from wmx_log_utils import load_wmx_log, rmse

## 2. WMX 로그 읽기와 전처리

최적화 중에는 Newton simulation이 고정된 `dt`를 사용한다. 따라서 로그의 시간이 단조 증가하는지, sample 간격이 충분히 일정한지 먼저 검사한다.

단위 변환은 로그를 읽는 시점에 한 번만 수행한다. Newton 모델과 목적함수 내부에서는 Position `[rad]`, Velocity `[rad/s]`, Torque `[N·m]`, Time `[s]`만 사용한다.

In [ ]:
def select_time_window(data, start_s=None, end_s=None):
    """로그의 연속 구간만 잘라낸다. sample을 건너뛰면 제어 주기가 바뀌므로 stride는 사용하지 않는다."""
    time = data["time"]
    start_s = time[0] if start_s is None else float(start_s)
    end_s = time[-1] + data["dt"] if end_s is None else float(end_s)
    mask = (time >= start_s) & (time < end_s)
    if np.count_nonzero(mask) < 2:
        raise ValueError("선택한 fitting 구간에 sample이 2개 미만입니다.")

    selected = {key: (value[mask].copy() if isinstance(value, np.ndarray) else value)
                for key, value in data.items()}
    selected["time"] -= selected["time"][0]
    return selected

In [ ]:
# 실제 로그에 맞게 경로와 단위 변환 계수를 수정한다.
FIT_LOG_PATH = None         # 예: Path("logs/wmx_fit_log.txt")
VALID_LOG_PATH = None       # 예: Path("logs/wmx_validation_log.txt")

LOG_OPTIONS = {
    "time_unit": "ms",
    "position_scale": 1.0,
    "velocity_scale": 1.0,
    "torque_scale": 1.0,
}

fit_data = (load_wmx_log(FIT_LOG_PATH, **LOG_OPTIONS) if FIT_LOG_PATH else None)
valid_data = (load_wmx_log(VALID_LOG_PATH, **LOG_OPTIONS) if VALID_LOG_PATH else None)

if fit_data is None:
    print("FIT_LOG_PATH를 지정한 뒤 이 셀을 다시 실행하세요.")
else:
    print(f"Fitting log: {len(fit_data['time'])} samples, dt={fit_data['dt']:.9g} s")
    # 필요하면 식별 정보가 풍부한 연속 구간만 선택한다.
    # fit_data = select_time_window(fit_data, start_s=1.0, end_s=6.0)

## 3. 파라미터 범위와 최적화 설정

서로 단위와 크기가 크게 다른 물리 파라미터를 그대로 L-BFGS-B에 전달하면 수치 미분의 step 크기가 불균형해질 수 있다. 여기서는 모든 연속 파라미터를 내부적으로 `[0, 1]`로 정규화하고, simulation 직전에 실제 물리 범위로 되돌린다. 여러 자릿수 범위를 탐색하는 양수 파라미터는 log scale로 변환한다.

`delay_steps`는 정수이고 반올림 때문에 목적함수가 미분 불가능하다. 따라서 연속 벡터에 억지로 포함하지 않고, 지정한 정수 후보마다 전역 탐색과 local refinement를 수행한 뒤 가장 작은 loss를 고른다.

아래 범위는 안전한 예시일 뿐 모터 사양을 반영한 확정 범위가 아니다. 정격 토크, 추정 관성, 제어기 설정값 등 알고 있는 정보를 이용해 가능한 한 현실적으로 좁혀야 한다.

In [ ]:
CONTINUOUS_PARAMETER_NAMES = (
    "kp", "ki", "kd", "integral_max",
    "effort_limit", "inertia", "viscous", "coulomb",
)

# 넓은 양수 범위에서 작은 값 쪽도 충분히 탐색하기 위한 log scale
LOG_SCALE_PARAMETERS = {"kp", "integral_max", "effort_limit", "inertia"}

PARAMETER_BOUNDS = {
    "kp": (0.1, 200.0),
    "ki": (0.0, 100.0),
    "kd": (0.0, 20.0),
    "integral_max": (1.0e-4, 10.0),
    "effort_limit": (0.1, 50.0),
    "inertia": (1.0e-5, 1.0),
    "viscous": (0.0, 5.0),
    "coulomb": (0.0, 10.0),
}

@dataclass
class FitConfig:
    parameter_bounds: dict = field(default_factory=lambda: dict(PARAMETER_BOUNDS))
    delay_candidates: tuple = tuple(range(0, 6))
    weights: dict = field(default_factory=lambda: {
        "position": 1.0, "velocity": 0.5, "torque": 0.5,
    })
    scale_floors: dict = field(default_factory=lambda: {
        "position": 1.0e-3, "velocity": 1.0e-2, "torque": 1.0e-2,
    })
    ignore_initial_samples: int = 0
    device: str = "cpu"
    seed: int = 42
    de_maxiter: int = 20
    de_popsize: int = 8
    de_tol: float = 1.0e-3
    local_maxiter: int = 150
    print_every: int = 50
    failure_penalty: float = 1.0e12


def validate_fit_config(config):
    missing = set(CONTINUOUS_PARAMETER_NAMES) - set(config.parameter_bounds)
    if missing:
        raise ValueError(f"parameter_bounds에 다음 항목이 없습니다: {sorted(missing)}")
    for name in CONTINUOUS_PARAMETER_NAMES:
        lower, upper = config.parameter_bounds[name]
        if not np.isfinite([lower, upper]).all() or lower >= upper:
            raise ValueError(f"{name}의 bound가 잘못되었습니다: {(lower, upper)}")
    if not config.delay_candidates or any(int(d) != d or d < 0 for d in config.delay_candidates):
        raise ValueError("delay_candidates는 0 이상의 정수 후보여야 합니다.")
    if config.ignore_initial_samples < 0:
        raise ValueError("ignore_initial_samples는 0 이상이어야 합니다.")
    if any(config.weights[name] < 0.0 for name in ("position", "velocity", "torque")):
        raise ValueError("signal weight는 0 이상이어야 합니다.")
    if sum(config.weights.values()) <= 0.0:
        raise ValueError("적어도 하나의 signal weight가 0보다 커야 합니다.")

## 4. Objective function

Position, Velocity, Torque는 단위와 수치 범위가 다르므로 raw RMSE를 바로 더하지 않는다. fitting 로그에서 각 신호의 표준편차를 구해 residual을 나눈 뒤, 사용자가 정한 weight로 합한다.

$$L(\theta)=\frac{\sum_s w_s\,\mathrm{mean}(((y_s-\hat{y}_s)/\sigma_s)^2)}{\sum_s w_s}$$

여기서 $s$는 Position, Velocity, Torque이다. 표준편차가 거의 0인 신호가 loss를 폭발시키지 않도록 단위별 최소 scale을 둔다. 최종 보고에서는 해석하기 쉬운 raw RMSE도 함께 계산한다.

In [ ]:
SIGNAL_MAP = {
    "position": ("feedback_position", "feedback_position"),
    "velocity": ("feedback_velocity", "feedback_velocity"),
    "torque": ("feedback_torque", "feedback_torque"),
}


def normalized_to_params(z, delay_steps, parameter_bounds):
    z = np.asarray(z, dtype=np.float64)
    if z.shape != (len(CONTINUOUS_PARAMETER_NAMES),):
        raise ValueError(f"normalized parameter shape가 잘못되었습니다: {z.shape}")

    values = {}
    for value, name in zip(z, CONTINUOUS_PARAMETER_NAMES):
        lower, upper = parameter_bounds[name]
        value = float(np.clip(value, 0.0, 1.0))
        if name in LOG_SCALE_PARAMETERS:
            if lower <= 0.0:
                raise ValueError(f"log scale parameter {name}의 lower bound는 0보다 커야 합니다.")
            values[name] = float(np.exp(np.log(lower) + value * np.log(upper / lower)))
        else:
            values[name] = float(lower + value * (upper - lower))
    values["delay_steps"] = int(delay_steps)
    return MotorParams(**values)


def calculate_signal_scales(data, scale_floors):
    return {
        signal: max(float(np.std(data[reference_key])), float(scale_floors[signal]))
        for signal, (reference_key, _) in SIGNAL_MAP.items()
    }


def run_model(data, params, device="cpu"):
    return simulate_motor(
        data["command_position"],
        data["dt"],
        params,
        device=device,
        use_viewer=False,
        initial_position=float(data["feedback_position"][0]),
        initial_velocity=float(data["feedback_velocity"][0]),
    )


def normalized_loss(data, simulation, scales, weights, ignore_initial_samples=0):
    start = int(ignore_initial_samples)
    if start >= len(data["time"]):
        raise ValueError("ignore_initial_samples가 전체 sample 수보다 크거나 같습니다.")

    weighted_loss = 0.0
    weight_sum = 0.0
    for signal, (reference_key, simulation_key) in SIGNAL_MAP.items():
        weight = float(weights[signal])
        if weight == 0.0:
            continue
        residual = (
            np.asarray(simulation[simulation_key][start:], dtype=np.float64)
            - np.asarray(data[reference_key][start:], dtype=np.float64)
        ) / scales[signal]
        weighted_loss += weight * float(np.mean(residual ** 2))
        weight_sum += weight
    return weighted_loss / weight_sum


def make_objective(data, config, scales):
    evaluations = {"count": 0, "best": np.inf}

    def objective(z, delay_steps):
        evaluations["count"] += 1
        try:
            params = normalized_to_params(z, delay_steps, config.parameter_bounds)
            simulation = run_model(data, params, device=config.device)
            loss = normalized_loss(
                data, simulation, scales, config.weights, config.ignore_initial_samples
            )
            if not np.isfinite(loss):
                loss = config.failure_penalty
        except (FloatingPointError, OverflowError, ValueError, RuntimeError):
            loss = config.failure_penalty

        evaluations["best"] = min(evaluations["best"], loss)
        if config.print_every and evaluations["count"] % config.print_every == 0:
            print(
                f"  evaluations={evaluations['count']:5d}, "
                f"current={loss:.6g}, best={evaluations['best']:.6g}"
            )
        return float(loss)

    return objective, evaluations

## 5. 1차 전역 탐색 + 2차 Local refinement

`differential_evolution`은 초기값 의존성이 낮아 넓은 영역을 찾는 데 적합하지만 simulation 호출 횟수가 많다. `de_maxiter`와 `de_popsize`를 작게 두고 pipeline을 먼저 검증한 뒤 늘리는 것이 좋다.

`L-BFGS-B`는 bound를 지키면서 전역 탐색 결과 주변을 정밀 탐색한다. 다만 토크 포화, 쿨롱 마찰, anti-windup 때문에 loss가 완전히 매끄럽지는 않다. local 결과가 전역 탐색 결과보다 나빠지면 이 함수는 더 좋은 전역 탐색 결과를 보존한다.

Newton/Warp 객체의 thread 안전성을 확인하기 전까지 `workers=1`로 실행한다.

In [ ]:
@dataclass
class FitResult:
    params: MotorParams
    loss: float
    normalized_vector: np.ndarray
    scales: dict
    candidates: list
    simulation: dict


def fit_motor_parameters(data, config=None):
    config = FitConfig() if config is None else config
    validate_fit_config(config)
    scales = calculate_signal_scales(data, config.scale_floors)
    unit_bounds = [(0.0, 1.0)] * len(CONTINUOUS_PARAMETER_NAMES)
    objective, evaluations = make_objective(data, config, scales)

    candidates = []
    best = None

    for delay_steps in config.delay_candidates:
        print(f"\n[delay_steps={delay_steps}] differential_evolution 시작")
        global_result = differential_evolution(
            lambda z: objective(z, delay_steps),
            bounds=unit_bounds,
            seed=config.seed,
            maxiter=config.de_maxiter,
            popsize=config.de_popsize,
            tol=config.de_tol,
            polish=False,
            workers=1,
            updating="immediate",
        )

        print(f"[delay_steps={delay_steps}] L-BFGS-B 시작")
        local_result = minimize(
            lambda z: objective(z, delay_steps),
            x0=global_result.x,
            method="L-BFGS-B",
            bounds=unit_bounds,
            options={"maxiter": config.local_maxiter, "ftol": 1.0e-12},
        )

        chosen_result = local_result if local_result.fun <= global_result.fun else global_result
        candidate = {
            "delay_steps": int(delay_steps),
            "global_loss": float(global_result.fun),
            "local_loss": float(local_result.fun),
            "local_success": bool(local_result.success),
            "loss": float(chosen_result.fun),
            "z": np.asarray(chosen_result.x, dtype=np.float64).copy(),
        }
        candidates.append(candidate)
        print(
            f"[delay_steps={delay_steps}] global={candidate['global_loss']:.6g}, "
            f"local={candidate['local_loss']:.6g}, selected={candidate['loss']:.6g}"
        )

        if best is None or candidate["loss"] < best["loss"]:
            best = candidate

    best_params = normalized_to_params(
        best["z"], best["delay_steps"], config.parameter_bounds
    )
    best_simulation = run_model(data, best_params, device=config.device)

    print(f"\n총 objective 평가 횟수: {evaluations['count']}")
    return FitResult(
        params=best_params,
        loss=float(best["loss"]),
        normalized_vector=best["z"],
        scales=scales,
        candidates=candidates,
        simulation=best_simulation,
    )

In [ ]:
# 먼저 작은 설정으로 전체 pipeline이 정상 동작하는지 확인한다.
config = FitConfig(
    parameter_bounds=dict(PARAMETER_BOUNDS),
    delay_candidates=tuple(range(0, 6)),
    de_maxiter=10,          # 본 fitting에서는 충분히 늘린다.
    de_popsize=6,
    local_maxiter=100,
    print_every=50,
)

fit_result = fit_motor_parameters(fit_data, config) if fit_data is not None else None

if fit_result is not None:
    print("\nBest normalized loss:", fit_result.loss)
    print("Best parameters:")
    for name, value in asdict(fit_result.params).items():
        print(f"  {name:>14s} = {value:.9g}" if isinstance(value, float) else f"  {name:>14s} = {value}")

## 6. 결과 평가와 시각화

최적화에 사용한 normalized loss만 보면 어느 신호가 좋아지고 나빠졌는지 알기 어렵다. Position `[rad]`, Velocity `[rad/s]`, Torque `[N·m]`의 RMSE를 따로 보고, 시간 파형과 residual의 구조도 확인한다.

residual이 무작위에 가깝지 않고 특정 방향·속도·가속 구간에서 반복되는 모양을 보이면, 단순한 파라미터 오차가 아니라 dead zone, backlash, 정지 마찰, torque filter 등 현재 모델에 없는 동역학일 수 있다.

In [ ]:
def evaluate_parameters(
    data, params, *, device="cpu", scales=None, weights=None, ignore_initial_samples=0
):
    simulation = run_model(data, params, device=device)
    metrics = {
        signal + "_rmse": rmse(data[reference_key], simulation[simulation_key])
        for signal, (reference_key, simulation_key) in SIGNAL_MAP.items()
    }
    if scales is not None and weights is not None:
        metrics["normalized_loss"] = normalized_loss(
            data, simulation, scales, weights, ignore_initial_samples
        )
    return {"simulation": simulation, "metrics": metrics}


def print_metrics(metrics, title="Metrics"):
    print(title)
    print(f"  Position RMSE : {metrics['position_rmse']:.6g} rad")
    print(f"  Velocity RMSE : {metrics['velocity_rmse']:.6g} rad/s")
    print(f"  Torque RMSE   : {metrics['torque_rmse']:.6g} N·m")
    if "normalized_loss" in metrics:
        print(f"  Normalized loss: {metrics['normalized_loss']:.6g}")


def plot_comparison(data, simulation, title):
    time = data["time"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(time, data["command_position"], "k--", label="Command")
    axes[0].plot(time, data["feedback_position"], label="WMX")
    axes[0].plot(time, simulation["feedback_position"], label="Newton")
    axes[0].set_ylabel("Position [rad]")

    axes[1].plot(time, data["feedback_velocity"], label="WMX")
    axes[1].plot(time, simulation["feedback_velocity"], label="Newton")
    axes[1].set_ylabel("Velocity [rad/s]")

    axes[2].plot(time, data["feedback_torque"], label="WMX")
    axes[2].plot(time, simulation["feedback_torque"], label="Newton")
    axes[2].set_ylabel("Torque [N·m]")
    axes[2].set_xlabel("Time [s]")

    for axis in axes:
        axis.grid(True)
        axis.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def plot_residuals(data, simulation, title):
    time = data["time"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    labels = {"position": "Position [rad]", "velocity": "Velocity [rad/s]", "torque": "Torque [N·m]"}
    for axis, (signal, (reference_key, simulation_key)) in zip(axes, SIGNAL_MAP.items()):
        residual = data[reference_key] - simulation[simulation_key]
        axis.plot(time, residual)
        axis.axhline(0.0, color="black", linewidth=0.8)
        axis.set_ylabel(labels[signal])
        axis.grid(True)
    axes[-1].set_xlabel("Time [s]")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

In [ ]:
if fit_result is not None:
    fit_evaluation = evaluate_parameters(
        fit_data,
        fit_result.params,
        device=config.device,
        scales=fit_result.scales,
        weights=config.weights,
        ignore_initial_samples=config.ignore_initial_samples,
    )
    print_metrics(fit_evaluation["metrics"], "Fitting log")
    plot_comparison(fit_data, fit_evaluation["simulation"], "Fitting log: WMX vs Newton")
    plot_residuals(fit_data, fit_evaluation["simulation"], "Fitting log residual (WMX - Newton)")

## 7. 3차 Validation: 다른 WMX 로그로 최종 확인

validation 로그에는 fitting 과정에서 얻은 파라미터를 그대로 적용한다. bounds, weight, 초기값 또는 파라미터를 validation 결과에 맞춰 다시 수정하면 그 로그도 사실상 fitting에 사용한 것이 되므로, 최종 성능 평가는 새로운 로그에서 다시 해야 한다.

validation의 normalized loss에는 fitting 로그에서 계산한 scale을 그대로 사용한다. 그래야 fitting과 validation 점수의 크기를 같은 기준으로 비교할 수 있다. raw RMSE는 각 신호의 실제 단위로 함께 확인한다.

In [ ]:
validation_evaluation = None

if fit_result is not None and valid_data is not None:
    validation_evaluation = evaluate_parameters(
        valid_data,
        fit_result.params,
        device=config.device,
        scales=fit_result.scales,
        weights=config.weights,
        ignore_initial_samples=config.ignore_initial_samples,
    )
    print_metrics(validation_evaluation["metrics"], "Validation log")
    plot_comparison(
        valid_data, validation_evaluation["simulation"], "Validation log: WMX vs Newton"
    )
    plot_residuals(
        valid_data, validation_evaluation["simulation"], "Validation residual (WMX - Newton)"
    )
elif VALID_LOG_PATH is None:
    print("VALID_LOG_PATH를 지정하면 별도 로그 validation을 실행할 수 있습니다.")

## 8. Fitted parameter 저장

최종 파라미터와 fitting 조건을 JSON으로 저장한다. 재현성을 위해 loss, signal scale, weight, parameter bounds와 원본 로그 경로도 함께 기록한다.

In [ ]:
def save_fit_result(file_path, fit_result, config, fit_data, validation_evaluation=None):
    payload = {
        "motor_params": asdict(fit_result.params),
        "fit_loss": fit_result.loss,
        "signal_scales": fit_result.scales,
        "weights": config.weights,
        "parameter_bounds": config.parameter_bounds,
        "delay_candidates": list(config.delay_candidates),
        "fit_log": fit_data.get("source"),
        "validation_metrics": (
            validation_evaluation["metrics"] if validation_evaluation is not None else None
        ),
    }
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"저장 완료: {file_path.resolve()}")


# fitting 완료 후 주석을 해제한다.
# save_fit_result(
#     "fitted_motor_params.json", fit_result, config, fit_data, validation_evaluation
# )

## 9. 결과 해석 체크리스트

- 여러 파라미터가 bound의 최솟값 또는 최댓값에 붙었다면 범위가 잘못되었거나 해당 파라미터를 로그에서 식별하기 어려운지 확인한다.
- 서로 다른 random seed에서 전혀 다른 파라미터가 비슷한 loss를 만들면 파라미터 간 상관관계 또는 비식별성 가능성이 크다.
- Position은 잘 맞지만 Torque가 맞지 않으면 Torque 단위/부호/필터와 drive가 보고하는 Torque의 정의를 먼저 확인한다.
- 정지 직전·방향 전환 구간의 residual이 크면 `sign(qd)`만 사용하는 현재 Coulomb 모델에 정지 마찰 또는 smooth friction 모델을 추가할지 검토한다.
- validation loss가 fitting loss보다 크게 나쁘면 한 로그에 과적합되었거나, 다른 운전 영역에 필요한 동역학이 모델에 빠져 있을 수 있다.
- 최종 결과를 채택하기 전에 다른 seed, 다른 command profile, 다른 시작 위치에서도 반복 검증한다.